# F1nalyze – Phase 1: Loading and profiling the data

**Project:** classification project · **Dataset:** F1nalyze – Formula 1 Datathon

Goal of Phase 1: load and profile all CSV files (shape, types, missing values, class distribution) and pick the best target variable.

> **Note 1 (technical):** `train.csv` is ~1.1 GB with 2.83M rows. We read it in chunks as text (`dtype=str`). Missing values are encoded as `\N`.

> **Note 2 (most important):** the raw data is a **broken join** and must be repaired — see section 2. All class distributions are computed **after the repair**.


In [1]:
import pandas as pd, numpy as np, os
pd.set_option('display.max_columns', 60)
DATA='data'
NA=['\\N','\\n']
FILES=['train.csv','validation.csv','test.csv','sample_submission.csv']


## 1. File shapes (raw)

| file | rows (raw) | columns | `position`? | year range |
|---|---:|---:|---|---|
| train.csv | 2,830,101 | 55 | YES | 1950–2013 |
| validation.csv | 353,762 | 55 | YES | 2013–2018 |
| test.csv | 352,928 | 53 | NO | 2018–2023 |
| sample_submission.csv | 18 | 2 | — (format) | — |

`test.csv` has neither `position` nor `positionText_y` → a competition (Kaggle) setup: we train on `train`, evaluate on `validation` (it has labels!), and `test` is used for the submission.


## 2. ⚠️ CRITICAL: the data is a broken join (Cartesian product) — and how to fix it

The row count is **artificially inflated**. There are far fewer real entries (driver × race, key `resultId`):

| file | rows | real entries (`resultId`) | inflation |
|---|---:|---:|---:|
| train | 2,830,101 | **22,089** (21,643 with an exact snapshot) | ~128× |
| validation | 353,762 | **1,836** | ~193× |
| test | 352,928 | **2,133** | ~165× |

Within a single `resultId` the race variables (`grid`, `circuitId`, `driverId`, `positionOrder`, `status`) are **constant**, but `position`, `points_y` and `driverStandingsId` take many different values — ~130 unrelated championship-standings snapshots were glued onto each race result. **Takeaway:** the raw row-level `position` is inconsistent; with honest pre-race features it is impossible to predict (the same feature row → many labels).

**Fix:** keep only the rows where `raceId_y == racerId` → exactly **1 row per `resultId`**. Then `position` becomes consistent. (For the ~2% of results with no exact snapshot, notebook 02 falls back to the nearest earlier snapshot.)


In [2]:
# Demonstrating the problem and the fix on validation (small file)
val=pd.read_csv(os.path.join(DATA,'validation.csv'), dtype=str, na_values=NA)
g=val.groupby('resultId')
print('rows:', len(val), '| real entries:', g.ngroups, f'| inflation ~{len(val)/g.ngroups:.0f}x')
print('distinct values per resultId (median):')
for col in ['positionOrder','grid','position','points_y','driverStandingsId']:
    print(f'  {col:<18}: {int(g[col].nunique().median())}  (1 = constant within an entry)')

# FIX
valf = val[val['raceId_y']==val['racerId']].copy()
print('\nAfter the fix: rows =', len(valf), '| unique resultId =', valf['resultId'].nunique())
assert len(valf)==valf['resultId'].nunique(), 'there should be 1 row per resultId'


rows: 353762 | real entries: 1836 | inflation ~193x
distinct values per resultId (median):
  positionOrder     : 1  (1 = constant within an entry)
  grid              : 1  (1 = constant within an entry)
  position          : 18  (1 = constant within an entry)
  points_y          : 84  (1 = constant within an entry)
  driverStandingsId : 184  (1 = constant within an entry)

After the fix: rows = 1835 | unique resultId = 1835


## 3. TIME-BASED split (not random!)

The year ranges are **disjoint**: train 1950–2013, validation 2013–2018, test 2018–2023. Consequences: (1) the model has to generalise forward in time; (2) distributions and missing values differ between files (different F1 eras); (3) you **must not** mix the files or use a random split.


## 4. Column types


In [3]:
def classify_type(s):
    s=s.dropna().head(50000)
    if len(s)==0: return 'empty (100% missing)'
    n=pd.to_numeric(s, errors='coerce')
    if n.notna().mean()>0.99: return 'int' if (n.dropna()%1==0).all() else 'float'
    return 'categorical/text'
types=pd.Series({col:classify_type(val[col]) for col in val.columns})
print(types.value_counts())


int                     24
categorical/text        19
empty (100% missing)    10
float                    2
Name: count, dtype: int64


**Result:** 24 integer, 2 float, 19 categorical/text, **10 columns that are 100% empty**. Some `*Id` columns are really categories (e.g. `circuitId`, `constructorId`).


## 5. Missing values


In [4]:
missing=val.isna().sum(); missing=missing[missing>0].sort_values(ascending=False)
print((missing/len(val)*100).round(1).astype(str)+' %')


fp2_date                 100.0 %
quali_date               100.0 %
sprint_time              100.0 %
sprint_date              100.0 %
quali_time               100.0 %
fp1_date                 100.0 %
fp1_time                 100.0 %
fp2_time                 100.0 %
fp3_date                 100.0 %
fp3_time                 100.0 %
timetaken_in_millisec     43.9 %
time_x                    43.9 %
position_x                17.6 %
max_speed                  4.6 %
fastestLapTime             4.6 %
fastestLap                 4.6 %
driver_num                 0.1 %
dtype: str


- **100% missing (10 columns — to drop):** `fp1/fp2/fp3_date+time`, `quali_date+time`, `sprint_date+time`.
- **~44%:** `time_x`, `timetaken_in_millisec`. **~18–40%:** `position_x` (DNF). **~5%:** `fastestLap`, `fastestLapTime`, `max_speed`.
- The remaining columns are essentially complete.


## 6. Target variable — distribution AFTER the repair (train)


In [5]:
def bucket(p):
    p=int(p)
    return 'win(1)' if p==1 else 'podium(2-3)' if p<=3 else 'points(4-10)' if p<=10 else 'rest(>10)'

use=['resultId','racerId','raceId_y','position','positionOrder']
posb={}; pob={}
for ch in pd.read_csv(os.path.join(DATA,'train.csv'), usecols=use, chunksize=500_000, dtype=str, na_values=NA):
    m=ch[ch['raceId_y']==ch['racerId']]
    for v in m['position'].dropna():      posb[bucket(v)]=posb.get(bucket(v),0)+1
    for v in m['positionOrder'].dropna(): pob[bucket(v)]=pob.get(bucket(v),0)+1
tp=sum(posb.values()); to=sum(pob.values())
print('position (championship standings):', {k:f'{100*v/tp:.1f}%' for k,v in sorted(posb.items())})
print('positionOrder (race result):', {k:f'{100*v/to:.1f}%' for k,v in sorted(pob.items())})


position (championship standings): {'podium(2-3)': '8.1%', 'points(4-10)': '26.9%', 'rest(>10)': '60.9%', 'win(1)': '4.1%'}
positionOrder (race result): {'podium(2-3)': '8.4%', 'points(4-10)': '29.1%', 'rest(>10)': '58.3%', 'win(1)': '4.2%'}


**Distribution of the 4 classes (train, after the repair):**

| class | `positionOrder` (result) | `position` (standings) |
|---|---:|---:|
| win (1) | 4.2% | 4.1% |
| podium (2–3) | 8.4% | 8.1% |
| points (4–10) | 29.1% | 26.9% |
| rest (>10) | 58.3% | 60.9% |

Imbalance ~14× (moderate). **Note:** the raw `position` on the broken data had 108 classes and an imbalance of ~170,000× — that was an **artefact** of the broken join, not reality.

**Semantics:** `position` = place in the championship standings after the race (there 'win' = championship leader). `positionOrder` = finishing position (there 'win' = winning the race).


## 7. Methodological pitfalls

**(A) Data leakage.** `positionText_y` is the target as text — always drop it. `points`, `position_x`, `positionOrder`, `rank` are post-race results. `points_y`/`wins` are the standings AFTER the race (cumulative) → they may only be used in a **lagged** version (the state before the race).

**(B) Repairing the join** (section 2) — mandatory before anything else.

**(C) Time-based split** (section 3) — validate on `validation`, time-aware CV, no random shuffling.

**(D) Junk columns:** `url_*`, `number`, `driver_num`, `driver_code` etc. — these are not features; `result_driver_standing` is the row ID (submission key).


## 8. Recommendation and findings (input to Phase 2/3)

Chosen approach: **repair the data → two targets (`positionOrder` and `position`) bucketed into 4 classes → compare**. Leakage-free features (pre-race + lagged historical). Models: logistic regression, random forest, gradient boosting; `class_weight='balanced'`. Metrics: macro-F1 (main), balanced accuracy, confusion matrix, per-class report. Extended EDA. Deliverable: a `.ipynb` notebook + a predictions file.

*Later update:* in notebook 02 the `position` target (Target B) was dropped — podium/points make no sense for the championship standings, and a trivial rule dominated it. The baseline to beat turned out to be the grid rule, not logistic regression.
